# Importing Libraries and Setup

In [1]:
import os

os.environ["HF_HOME"] = f"/rs1/researchers/a/amallav/models/hf_home"

os.environ["HF_HUB_CACHE"] = os.path.join(os.environ["HF_HOME"], "hub")
os.environ["HF_HUB_OFFLINE"] = "1"   # only after cache is populated

In [2]:
import json #Used later to store the top-k predictions as a JSON string in the final CSV
from pathlib import Path
import pandas as pd
import torch
from bioclip import TreeOfLifeClassifier, Rank #TreeOfLifeClassifier = the classifier that predicts biological taxa; Rank = tells the classifier what taxonomic level you want

/usr/local/usrapps/ftrscape/snair3/env_AiPipeline/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# os.environ["PROJECT_DIR"] = "/rs1/researchers/a/amallav/"
# print(os.environ["PROJECT_DIR"])
# os.environ["OUTPUTS_DIR"] = "/rs1/researchers/a/amallav/results"
# print(os.environ["OUTPUTS_DIR"])
# os.environ["OUTPUTS_CP_DIR"] = "/rs1/researchers/a/amallav/results/outputs_crop"
# print(os.environ["OUTPUTS_CP_DIR"])
# os.environ["OUTPUTS_BCCP_DIR"] = "/rs1/researchers/a/amallav/results/outputs_bioclip_crop"
# print(os.environ["OUTPUTS_BCCP_DIR"])

In [ ]:
PROJECT_DIR = Path(os.environ.get("PROJECT_DIR", "/rs1/researchers/a/amallav/")).resolve()
OUTPUTS_DIR=Path(os.environ.get("OUTPUTS_DIR", "/rs1/researchers/a/amallav/results")).resolve()

OUTPUTS_CP_DIR = Path(os.environ.get("OUTPUTS_CP_DIR", OUTPUTS_DIR / "outputs_crop_round2")).resolve()

OUTPUTS_BCCP_DIR = Path(os.environ.get("OUTPUTS_BCCP_DIR", OUTPUTS_DIR / "outputs_bioclip_crop2")).resolve()
OUTPUTS_BCCP_DIR.mkdir(parents=True, exist_ok=True)

# CROP_OUT = PROJECT_DIR / "outputs_crop"
META_CSV = OUTPUTS_CP_DIR / "cropped_metadata.csv" #This is the metadata file that tells us what crop files exist.
OUT_CSV = OUTPUTS_BCCP_DIR / "bioclip_species_predictions.csv"
output_csv = OUTPUTS_BCCP_DIR / "predictions.csv"

print("PROJECT_DIR:", PROJECT_DIR)
print("OUTPUTS_DIR:", OUTPUTS_DIR, "| exists:", OUTPUTS_DIR.exists())
print("OUTPUTS_CP_DIR:", OUTPUTS_CP_DIR, "| exists:", OUTPUTS_CP_DIR.exists())
print("OUTPUTS_BCCP_DIR:", OUTPUTS_BCCP_DIR, "| exists:", OUTPUTS_BCCP_DIR.exists())
print("META_CSV    :", META_CSV, "| exists:", META_CSV.exists())
print("OUT_CSV     :", OUT_CSV, "| exists:", OUT_CSV.exists())
print("output_csv     :", output_csv, "| exists:", output_csv.exists())

PROJECT_DIR: /rs1/researchers/a/amallav
OUTPUTS_DIR: /rs1/researchers/a/amallav/results | exists: True
OUTPUTS_CP_DIR: /rs1/researchers/a/amallav/results/outputs_crop | exists: True
OUTPUTS_BCCP_DIR: /rs1/researchers/a/amallav/results/outputs_bioclip_crop | exists: True
META_CSV    : /rs1/researchers/a/amallav/results/outputs_crop/cropped_metadata.csv | exists: True
OUT_CSV     : /rs1/researchers/a/amallav/results/outputs_bioclip_crop/bioclip_species_predictions.csv | exists: False
output_csv     : /rs1/researchers/a/amallav/results/outputs_bioclip_crop/predictions.csv | exists: True


# Load Bioclip 

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [6]:
# model_dir = "/share/ftrscape/{}/models/bioclip".format(__import__("os").environ["USER"])

In [7]:
MODEL_STR = "hf-hub:imageomics/bioclip-2"

TOP_K = 5 #top 5 species predictions for each image
BATCH_SIZE = 1 #This controls how many cropped images are processed at once. Might have to tune this parameter

classifier = TreeOfLifeClassifier(
    device=device,
    model_str=MODEL_STR,
)

print("Device   :", device)
print("Model    :", MODEL_STR)
print("Top-k    :", TOP_K)
print("Batch size:", BATCH_SIZE)

Device   : cuda
Model    : hf-hub:imageomics/bioclip-2
Top-k    : 5
Batch size: 1


# Load CSV

In [12]:
assert META_CSV.exists(), f"metadata.csv not found: {META_CSV}"

meta = pd.read_csv(META_CSV)

required_cols = {"crop_file_path", "crop_file"}
missing_cols = required_cols - set(meta.columns)
assert not missing_cols, f"metadata.csv is missing columns: {missing_cols}"

In [13]:
meta

,original_image,image_path,box_num,image_id,crop_file,crop_file_path,x_min,y_min,x_max,y_max
0,obs_10003229_photo_114145431.jpg,/rs1/researchers/a/amallav/image_dataset/obs_1...,1,obs_10003229_photo_114145431,obs_10003229_photo_114145431_box1.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,192,1,1053,987
1,obs_10003229_photo_114145431.jpg,/rs1/researchers/a/amallav/image_dataset/obs_1...,2,obs_10003229_photo_114145431,obs_10003229_photo_114145431_box2.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,277,94,1030,783
2,obs_10003229_photo_114145431.jpg,/rs1/researchers/a/amallav/image_dataset/obs_1...,3,obs_10003229_photo_114145431,obs_10003229_photo_114145431_box3.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,281,95,927,772
3,obs_10003229_photo_114145431.jpg,/rs1/researchers/a/amallav/image_dataset/obs_1...,4,obs_10003229_photo_114145431,obs_10003229_photo_114145431_box4.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,185,6,1055,786
4,obs_10003229_photo_114145431.jpg,/rs1/researchers/a/amallav/image_dataset/obs_1...,5,obs_10003229_photo_114145431,obs_10003229_photo_114145431_box5.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,284,98,901,764
...,...,...,...,...,...,...,...,...,...,...
74744,obs_99990956_photo_166796339.jpg,/rs1/researchers/a/amallav/image_dataset/obs_9...,2,obs_99990956_photo_166796339,obs_99990956_photo_166796339_box2.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,180,46,1681,1733
74745,obs_99990956_photo_166796339.jpg,/rs1/researchers/a/amallav/image_dataset/obs_9...,3,obs_99990956_photo_166796339,obs_99990956_photo_166796339_box3.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,191,613,1599,1723
74746,obs_99990956_photo_166796339.jpg,/rs1/researchers/a/amallav/image_dataset/obs_9...,4,obs_99990956_photo_166796339,obs_99990956_photo_166796339_box4.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,192,585,1598,1719
74747,obs_99990956_photo_166796345.jpg,/rs1/researchers/a/amallav/image_dataset/obs_9...,1,obs_99990956_photo_166796345,obs_99990956_photo_166796345_box1.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,660,94,1644,1436


In [14]:
# #converts paths into absolute paths.
# def resolve_path(p):
#     if pd.isna(p):
#         return None
#     p = str(p).strip()
#     if os.path.isabs(p):
#         return p
#     return str((PROJECT_DIR / p).resolve())

# meta["masked_crop_path"] = meta["masked_crop_path"].apply(resolve_path)

In [15]:
meta = meta[meta["crop_file_path"].notna()].copy() #remove rows where crop path is missing
meta["crop_exists"] = meta["crop_file_path"].apply(os.path.exists)

missing_count = (~meta["crop_exists"]).sum()
if missing_count > 0:
    print(f"Skipping {missing_count} rows because crop file does not exist.")

meta = meta[meta["crop_exists"]].copy()  #only rows where the crop file actually exists.



Skipping 3 rows because crop file does not exist.


In [16]:
keep_cols = ["crop_file", "crop_file_path", "image_id"]
if "box_num" in meta.columns:
    keep_cols.append("box_num")

crop_df = meta[keep_cols].drop_duplicates(subset=["crop_file_path"]).reset_index(drop=True) #creates a clean crop table without deduplicates

print("Total valid cropped images:", len(crop_df))
crop_df.head()

Total valid cropped images: 74746


,crop_file,crop_file_path,image_id,box_num
0,obs_10003229_photo_114145431_box1.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,obs_10003229_photo_114145431,1
1,obs_10003229_photo_114145431_box2.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,obs_10003229_photo_114145431,2
2,obs_10003229_photo_114145431_box3.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,obs_10003229_photo_114145431,3
3,obs_10003229_photo_114145431_box4.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,obs_10003229_photo_114145431,4
4,obs_10003229_photo_114145431_box5.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,obs_10003229_photo_114145431,5


# Run species classification on all cropped images

In [13]:
# test = classifier.predict(
#     images="/gpfs_common/share03/ftrscape/snair3/wew_notebooks/outputs_crop/obs_250956763_photo_448935283_box2_copy.JPG",
#     rank=Rank.SPECIES, #Predict at species level.
#     k=TOP_K, #Return the top 5 predictions for each image.
#     batch_size=BATCH_SIZE, #Process 1 image at a time.
# )
# test_df=pd.DataFrame(test)

# # print("Total prediction rows:", len(pred_df))
# test_df.head()

In [13]:
# crop_paths = crop_df["crop_file_path"].tolist() #Collect all crop image paths into a list.

# predictions = classifier.predict(
#     images=crop_paths, #Pass the cropped image files to the model.
#     rank=Rank.SPECIES, #Predict at species level.
#     k=TOP_K, #Return the top 5 predictions for each image.
#     batch_size=BATCH_SIZE, #Process 1 image at a time.
# )

# pred_df = pd.DataFrame(predictions) #Turns the prediction output into a table.

# print("Total prediction rows:", len(pred_df))
# pred_df.head()

100%|██████████| 318/318 [00:18<00:00, 17.45images/s]


Total prediction rows: 1590


,file_name,kingdom,phylum,class,order,family,genus,species_epithet,species,common_name,score
0,/rs1/researchers/a/amallav/results/outputs_cro...,Animalia,Echinodermata,Asteroidea,Valvatida,Odontasteridae,Odontaster,penicillatus,Odontaster penicillatus,,0.060668
1,/rs1/researchers/a/amallav/results/outputs_cro...,Animalia,Echinodermata,Asteroidea,Valvatida,Goniasteridae,Ceramaster,japonicus,Ceramaster japonicus,Bat star,0.060482
2,/rs1/researchers/a/amallav/results/outputs_cro...,Animalia,Echinodermata,Asteroidea,Valvatida,Goniasteridae,Stellaster,princeps,Stellaster princeps,,0.055059
3,/rs1/researchers/a/amallav/results/outputs_cro...,Animalia,Echinodermata,Asteroidea,Valvatida,Asteropseidae,Dermasterias,imbricata,Dermasterias imbricata,,0.044492
4,/rs1/researchers/a/amallav/results/outputs_cro...,Animalia,Echinodermata,Asteroidea,Paxillosida,Astropectinidae,Dipsacaster,pretiosus,Dipsacaster pretiosus,,0.042861


In [14]:



# Create CSV with header only once

if not os.path.exists(output_csv):
    empty_df = pd.DataFrame(columns=[
        "file_name",
        "kingdom",
        "phylum",
        "class",
        "order",
        "family",
        "genus",
        "species_epithet",
        "species",
        "common_name",
        "score",
        "path",
        "id"
    ])
    empty_df.to_csv(output_csv, index=False)


for idx, row in crop_df.iterrows():
    crop_path = row["crop_file_path"]

    # Run prediction for ONE image
    prediction = classifier.predict(
        images=[crop_path],           # single image
        rank=Rank.SPECIES,
        k=TOP_K,
        batch_size=1
    )

    # Convert to DataFrame
    pred_df = pd.DataFrame(prediction)

    # (Optional) keep track of source image
    pred_df["crop_file_path"] = crop_path
    pred_df["crop_index"] = idx

    # Append to CSV (no header after first time)
    pred_df.to_csv(
        output_csv,
        mode="a",
        header=not os.path.getsize(output_csv),
        index=False
    )

    print(f"✅ Processed {crop_path} - {idx + 1}/{len(crop_df)}")

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.01images/s]


✅ Processed /rs1/researchers/a/amallav/results/outputs_crop/obs_10003229_photo_114145431_box1.jpg - 1/74746


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 22.83images/s]


✅ Processed /rs1/researchers/a/amallav/results/outputs_crop/obs_10003229_photo_114145431_box2.jpg - 2/74746


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 35.66images/s]


...

✅ Processed /rs1/researchers/a/amallav/results/outputs_crop/obs_110061275_photo_185605780_box5.jpg - 3459/74746


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 17.21images/s]


✅ Processed /rs1/researchers/a/amallav/results/outputs_crop/obs_110162051_photo_185792696_box1.jpg - 3460/74746


  0%|                                                                                                                              | 0/1 [00:00<?, ?images/s]


KeyboardInterrupt: 

In [9]:
pred_df = pd.read_csv(output_csv)
pred_df

,file_name,kingdom,phylum,class,order,family,genus,species_epithet,species,common_name,score,path,id
0,/rs1/researchers/a/amallav/results/outputs_cro...,Animalia,Arthropoda,Insecta,Lepidoptera,Erebidae,Cerocala,contraria,Cerocala contraria,NaN,0.178396,/rs1/researchers/a/amallav/results/outputs_cro...,0
1,/rs1/researchers/a/amallav/results/outputs_cro...,Animalia,Arthropoda,Insecta,Lepidoptera,Erebidae,Chrysorithrum,flavomaculata,Chrysorithrum flavomaculata,NaN,0.134323,/rs1/researchers/a/amallav/results/outputs_cro...,0
2,/rs1/researchers/a/amallav/results/outputs_cro...,Animalia,Arthropoda,Insecta,Lepidoptera,Erebidae,Ulotrichopus,marmoratus,Ulotrichopus marmoratus,NaN,0.066452,/rs1/researchers/a/amallav/results/outputs_cro...,0
3,/rs1/researchers/a/amallav/results/outputs_cro...,Animalia,Arthropoda,Insecta,Lepidoptera,Erebidae,Eudocima,divitiosa,Eudocima divitiosa,NaN,0.058108,/rs1/researchers/a/amallav/results/outputs_cro...,0
4,/rs1/researchers/a/amallav/results/outputs_cro...,Animalia,Arthropoda,Insecta,Lepidoptera,Erebidae,Eudocima,colubra,Eudocima colubra,NaN,0.032328,/rs1/researchers/a/amallav/results/outputs_cro...,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
373725,/rs1/researchers/a/amallav/results/outputs_cro...,Animalia,Chordata,Aves,Musophagiformes,Musophagidae,Tauraco,porphyreolophus,Tauraco porphyreolophus,Purple-crested Turaco,0.921519,/rs1/researchers/a/amallav/results/outputs_cro...,74745
373726,/rs1/researchers/a/amallav/results/outputs_cro...,Animalia,Chordata,Aves,Musophagiformes,Musophagidae,Tauraco,hartlaubi,Tauraco hartlaubi,Hartlaub's Turaco,0.019476,/rs1/researchers/a/amallav/results/outputs_cro...,74745
373727,/rs1/researchers/a/amallav/results/outputs_cro...,Animalia,Chordata,Aves,Musophagiformes,Musophagidae,Tauraco,erythrolophus,Tauraco erythrolophus,Red-crested Turaco,0.007151,/rs1/researchers/a/amallav/results/outputs_cro...,74745
373728,/rs1/researchers/a/amallav/results/outputs_cro...,Animalia,Chordata,Aves,Musophagiformes,Musophagidae,Tauraco,fischeri,Tauraco fischeri,Fischer's turaco,0.005640,/rs1/researchers/a/amallav/results/outputs_cro...,74745


# Convert top-k predictions into one row per cropped image

In [10]:
assert "file_name" in pred_df.columns, "Expected 'file_name' in prediction output"
assert "score" in pred_df.columns, "Expected 'score' in prediction output"
assert "species" in pred_df.columns, "Expected 'species' in prediction output"

In [15]:
# pred_df

,file_name,kingdom,phylum,class,order,family,genus,species_epithet,species,common_name,score
0,/rs1/researchers/a/amallav/results/outputs_cro...,Animalia,Echinodermata,Asteroidea,Valvatida,Odontasteridae,Odontaster,penicillatus,Odontaster penicillatus,,0.060668
1,/rs1/researchers/a/amallav/results/outputs_cro...,Animalia,Echinodermata,Asteroidea,Valvatida,Goniasteridae,Ceramaster,japonicus,Ceramaster japonicus,Bat star,0.060482
2,/rs1/researchers/a/amallav/results/outputs_cro...,Animalia,Echinodermata,Asteroidea,Valvatida,Goniasteridae,Stellaster,princeps,Stellaster princeps,,0.055059
3,/rs1/researchers/a/amallav/results/outputs_cro...,Animalia,Echinodermata,Asteroidea,Valvatida,Asteropseidae,Dermasterias,imbricata,Dermasterias imbricata,,0.044492
4,/rs1/researchers/a/amallav/results/outputs_cro...,Animalia,Echinodermata,Asteroidea,Paxillosida,Astropectinidae,Dipsacaster,pretiosus,Dipsacaster pretiosus,,0.042861
...,...,...,...,...,...,...,...,...,...,...,...
1585,/rs1/researchers/a/amallav/results/outputs_cro...,Animalia,Chordata,Aves,Anseriformes,Anatidae,Bucephala,albeola,Bucephala albeola,Bufflehead,0.786600
1586,/rs1/researchers/a/amallav/results/outputs_cro...,Animalia,Chordata,,Siluriformes,Ictaluridae,Ameiurus,nebulosus,Ameiurus nebulosus,Brown bullhead,0.006554
1587,/rs1/researchers/a/amallav/results/outputs_cro...,Animalia,Chordata,,Siluriformes,Ictaluridae,Ameiurus,catus,Ameiurus catus,White catfish,0.005098
1588,/rs1/researchers/a/amallav/results/outputs_cro...,Animalia,Chordata,,Anguilliformes,Anguillidae,Anguilla,rostrata,Anguilla rostrata,American eel,0.004807


In [17]:
summary_rows = []

for crop_path, group in pred_df.groupby("file_name", sort=False):
    group = group.sort_values("score", ascending=False).reset_index(drop=True) #This ensures highest-scoring prediction comes first.
    top1 = group.iloc[0]
    top2 = group.iloc[1]
    top3 = group.iloc[2]
    top4 = group.iloc[3]
    top5 = group.iloc[4]

    topk_records = group[["species", "common_name", "score"]].to_dict(orient="records") #keeps all top-k predictions, but only the most useful columns

    #summary row: one final record per crop, best prediction at top level and top-k stored as JSON
    row = {
        "crop_file_path": crop_path,
        "top1_species": top1.get("species"),
        "top2_species": top2.get("species"),
        "top3_species": top3.get("species"),
        "top4_species": top4.get("species"),
        "top5_species": top5.get("species"),
        "top1_common_name": top1.get("common_name"),
        "top1_score": top1.get("score"),
        "topk_predictions_json": json.dumps(topk_records, ensure_ascii=False),
    }
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

final_df = crop_df.merge(summary_df, on="crop_file_path", how="left") #why left? keeps all crops from crop_df, even if somehow a prediction row is missing.

# final_df["low_confidence_flag"] = final_df["top1_score"] < 0.20

print("Final rows:", len(final_df))
final_df.head()

Final rows: 74746


,crop_file,crop_file_path,image_id,box_num,top1_species,top2_species,top3_species,top4_species,top5_species,top1_common_name,top1_score,topk_predictions_json
0,obs_10003229_photo_114145431_box1.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,obs_10003229_photo_114145431,1,Cerocala contraria,Chrysorithrum flavomaculata,Ulotrichopus marmoratus,Eudocima divitiosa,Eudocima colubra,NaN,0.178396,"[{""species"": ""Cerocala contraria"", ""common_nam..."
1,obs_10003229_photo_114145431_box2.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,obs_10003229_photo_114145431,2,Eudocima divitiosa,Eudocima colubra,Cerocala contraria,Eudocima jordani,Eudocima imperator,NaN,0.375307,"[{""species"": ""Eudocima divitiosa"", ""common_nam..."
2,obs_10003229_photo_114145431_box3.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,obs_10003229_photo_114145431,3,Eudocima divitiosa,Eudocima jordani,Eudocima imperator,Eudocima colubra,Cerocala contraria,NaN,0.376554,"[{""species"": ""Eudocima divitiosa"", ""common_nam..."
3,obs_10003229_photo_114145431_box4.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,obs_10003229_photo_114145431,4,Eudocima divitiosa,Cerocala contraria,Chrysorithrum flavomaculata,Eudocima colubra,Megistoclisma ribbei,NaN,0.316769,"[{""species"": ""Eudocima divitiosa"", ""common_nam..."
4,obs_10003229_photo_114145431_box5.jpg,/rs1/researchers/a/amallav/results/outputs_cro...,obs_10003229_photo_114145431,5,Eudocima divitiosa,Eudocima imperator,Ulotrichopus marmoratus,Chrysorithrum flavomaculata,Eudocima colubra,NaN,0.284782,"[{""species"": ""Eudocima divitiosa"", ""common_nam..."


In [ ]:
summary_rows = []

for crop_path, group in pred_df.groupby("file_name", sort=False):
    group = group.sort_values("score", ascending=False).reset_index(drop=True) #This ensures highest-scoring prediction comes first.
    top1 = group.iloc[0]
    top2 = group.iloc[1]
    top3 = group.iloc[2]
    top4 = group.iloc[3]
    top5 = group.iloc[4]

    topk_records = group[["species", "common_name", "score"]].to_dict(orient="records") #keeps all top-k predictions, but only the most useful columns

    #summary row: one final record per crop, best prediction at top level and top-k stored as JSON
    for i in range(5):
        no=i+1
        no=str(no)
        val="top"+no+"_species"
        sc="top"+no+"_score"
        cc="top"+no+"_common_name"
        row = {
            "crop_file_path": crop_path,
            # val:group.iloc[i].get("species"),
            "species": group.iloc[i].get("species"),
            # "top2_species": top2.get("species"),
            # "top3_species": top3.get("species"),
            # "top4_species": top4.get("species"),
            # "top5_species": top5.get("species"),
            "common_name": group.iloc[i].get("common_name"),
            "kingdom": group.iloc[i].get("kingdom"),
            "phylum": group.iloc[i].get("phylum"),
            "class": group.iloc[i].get("class"),
            "order": group.iloc[i].get("order"),
            "family": group.iloc[i].get("family"),
            "genus": group.iloc[i].get("genus"),
            "score": group.iloc[i].get("score"),
            "top_no":no,
            "topk_predictions_json": json.dumps(topk_records, ensure_ascii=False),
        }
        summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

full_df = crop_df.merge(summary_df, on="crop_file_path", how="left") #why left? keeps all crops from crop_df, even if somehow a prediction row is missing.

# final_df["low_confidence_flag"] = final_df["top1_score"] < 0.20

print("Final rows:", len(full_df))
full_df.head()

# Save final output CSV

In [ ]:
OUT_CSV = OUTPUTS_BCCP_DIR / "bioclip_species_predictions.csv"
final_df.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)

Saved: /rs1/researchers/a/amallav/results/outputs_bioclip_crop/bioclip_species_predictions_2.csv


In [ ]:
OUT_CSV = OUTPUTS_BCCP_DIR / "results_crop.csv"
full_df.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)

Saved: /rs1/researchers/a/amallav/results/outputs_bioclip_crop/results_crop_2.csv
